# CELL-FM — OpenCell virtual staining

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BoHuangLab/CELL-FM/blob/master/notebooks/opencell_vs.ipynb)

Given a nucleus and an amino-acid sequence, CELL-FM predicts what the protein image would 
be if you tagged and imaged it. The OpenCell checkpoint was fine-tuned with all OpenCell proteins.

---

**You need a GPU runtime.** *Runtime → Change runtime type → T4 GPU*. Setup downloads about
6 GB once — 3.7 GB of CELL-FM weights and 2.3 GB for the ESM-C protein encoder. Generation
is quick by comparison: the defaults measured about two minutes end to end on an A40, so
budget something under ten on a Colab T4.

| | |
| --- | --- |
| Weights | [huggingface.co/BoHuangLab/CELL-FM](https://huggingface.co/BoHuangLab/CELL-FM) |
| Code | [github.com/BoHuangLab/CELL-FM](https://github.com/BoHuangLab/CELL-FM) |

## 1 · Setup

Check the runtime, install what Colab does not ship, fetch the code, weights, gene table
and reference cells, then build the model. Run them once, in order.

In [ ]:
import subprocess
import sys

print("python  ", sys.version.split()[0])
try:
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True,
    ).stdout.strip()
except FileNotFoundError:
    gpu = ""
print("gpu     ", gpu if gpu else
      "NONE — Runtime > Change runtime type > T4 GPU, then rerun this cell")


In [ ]:
# Colab already ships torch, numpy, pandas and matplotlib; this adds the rest of the stack.
# Two choices below look odd and are deliberate:
#
#   --no-deps on esm   its metadata requires torchtext, which has no wheel past Python
#                      3.11 and would drag torch backwards. Nothing on the ESM-C code
#                      path imports it — the second line is what the import closure
#                      actually needs, esm's other bounds included.
#   "transformers<4.48.2"  esm's own bound. It is behavioural, not cosmetic: 4.47 replaced
#                      the special-token properties on PreTrainedTokenizer with a
#                      _special_tokens_map served through __getattr__, and a tokenizer that
#                      misses the change returns mask_token = None, which kills generation
#                      inside esm/utils/encoding.py with "replace() argument 2 must be str,
#                      not None". esm 3.2 adapted to it; the bound is where that adaptation
#                      stops being tested.
#   no flash-attn      without it ESM-C falls back to a pure-torch rotary embedding,
#                      verified to give identical results, and skips a CUDA build.
#
# esm 3.1.4 used to be pinned here, and it forced a much worse install: it requires
# biotite==0.41.2, biotite 0.41 requires numpy<2, and biotite sits on the ESM-C import
# path — so the whole stack came down to NumPy 1.x. Under Colab's Python 3.13 neither
# numpy 1.26 nor biotite 0.41 has a wheel, so pip built both from source, and the numpy
# downgrade collided with every preinstalled Colab package that wants NumPy 2. esm 3.2
# moved to biotite>=1.0, which is NumPy 2 clean, and its ESM-C modules are byte-identical
# to 3.1.4's, so the checkpoint's tensor names are unchanged.
import sys

%pip install -q --no-deps --no-warn-conflicts "esm==3.2.1.post1"
%pip install -q --no-warn-conflicts "transformers<4.48.2" "diffusers==0.31.0" torchdiffeq loguru accelerate "biotite>=1.0" biopython msgpack-numpy cloudpathlib tenacity brotli zstd attrs einops tifffile

import importlib.metadata as md

# --no-warn-conflicts silences pip's post-install report, which here says only two things
# and both are expected:
#
#   "esm requires torchtext, which is not installed"   left unenforced on purpose by
#       --no-deps: nothing on the ESM-C code path imports it, and it has no wheel for this
#       Python. esm's transformers bound, by contrast, IS enforced above — that one is real.
#   "gradio requires huggingface-hub>=1.16"   transformers < 4.48.2 wants hub < 1.0 and
#       Colab preinstalls a gradio that wants a newer one. Nothing here imports gradio.
#
# The flag is safe here because it is not what verifies the install: the check below reads
# the installed versions, and the model cell asserts the tokenizer behaviour the pin exists
# to protect. Both are stronger than pip's declarative check, which only compares metadata.
print("\nversions")
for pkg in ("torch", "numpy", "pandas", "transformers", "diffusers", "esm", "biotite"):
    try:
        print(f"  {pkg:14s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"  {pkg:14s} MISSING")


def _version(pkg):
    """(major, minor, patch) for comparison; 4.48 sorts below 4.48.2, as it should."""
    return tuple(int(x) for x in md.version(pkg).split(".")[:3] if x.isdigit())


ready = _version("transformers") < (4, 48, 2)
print(f"\ntransformers {md.version('transformers')} < 4.48.2:",
      "yes — the imports below will work" if ready else
      "NO — the next cell will fail; rerun this one")

# A pin landing on disk is not the same as this kernel using it. If an earlier attempt got
# as far as the next cell then transformers is imported, and pip downgrades it underneath a
# kernel that goes on holding the old module in memory. So compare what is imported against
# what is installed and restart if they disagree. Expected, not a crash: Colab reconnects on
# its own and you carry on from the next cell, without rerunning this one.
stale = []
for pkg in ("transformers", "tokenizers", "huggingface_hub", "biotite"):
    loaded = getattr(sys.modules.get(pkg), "__version__", None)
    try:
        installed = md.version(pkg)
    except md.PackageNotFoundError:
        continue
    if loaded is not None and loaded != installed:
        stale.append(f"{pkg} {loaded} -> {installed}")

if stale:
    import os, time
    print("\nRestarting the kernel so these take effect — expected, not a crash:")
    for line in stale:
        print("   ", line)
    print("Colab reconnects by itself; continue from the next cell.")
    time.sleep(1)
    os.kill(os.getpid(), 9)


In [ ]:
import os
import sys
import time

# transformers probes for a TensorFlow backend at import time and imports it if present.
# Colab ships TensorFlow, nothing here uses it, and loading it costs seconds for nothing.
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers
from huggingface_hub import hf_hub_download, snapshot_download

# esm 3.2 needs transformers < 4.48.2. Past that the special-token plumbing it adapted to
# moves again, tokenizer.mask_token comes back None, and generation dies inside esm on
# "replace() argument 2 must be str, not None" rather than at import.
if tuple(int(x) for x in transformers.__version__.split(".")[:3]) >= (4, 48, 2):
    raise RuntimeError(
        f"This kernel has transformers {transformers.__version__} loaded; esm 3.2 needs "
        "< 4.48.2. If the install cell above already pins it, the kernel is holding the "
        "old module: Runtime > Restart session, then run the cells again from the top."
    )

# The model code comes from the public Space, the same snapshot the other notebooks use —
# CELLFMModel.sequence_to_image is already in that vendored subset, so this application
# needs no code of its own, only its own weights and assets.
CODE = snapshot_download(repo_id="BoHuangLab/CELL-FM", repo_type="space")
sys.path.insert(0, CODE)

WEIGHTS_REPO = "BoHuangLab/CELL-FM"
opencell = lambda name: hf_hub_download(repo_id=WEIGHTS_REPO, filename=f"opencell/{name}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("code   ", CODE)
print("weights", WEIGHTS_REPO, "opencell/")
print("device ", DEVICE)

# One palette for every figure below.
INK, MUTED, SURFACE = "#0b0b0b", "#52514e", "#fcfcfb"
NUCLEAR, CYTO, MARK = "#2a78d6", "#eb6834", "#e53935"
plt.rcParams.update({
    "figure.dpi": 120, "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "font.size": 9,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False,
})

In [ ]:
from cell_fm.criterions.cell_fm.unidiffuser import UniDiffCriterions
from cell_fm.models.cell_fm.cell_fm_config import CELLFMConfig
from cell_fm.models.cell_fm.cell_fm_model import CELLFMModel
from esm.tokenization.sequence_tokenizer import EsmSequenceTokenizer
from esm.utils import encoding

# Hyperparameters transcribed from scripts/cell_fm/evaluate_virtual_staining_opencell.sh,
# and checked against finetune_opencell/cellfm_vs/checkpoint-100000/config.json.
#
# This is the OpenCell virtual-staining model, fine-tuned from the pretrained one. It
# differs from the HPA generator the NLS notebook loads in one architectural way you can
# see from here: cell_image is "nucl" alone rather than "nucl,er,mt", so cond_conv expects
# a ONE-channel conditioning image. Handing it three fails inside a Conv2d.
#
# infer=True is not optional: load_pretrained_weights is a no-op without it, and the model
# would run on its random initialisation rather than fail.
config = CELLFMConfig(
    img_resize=256,
    img_crop_size=256,
    cell_image="nucl",
    test_cell_image="nucl",
    seq_zero_mask_ratio=0.0,
    path_type="Linear",
    prediction="velocity",
    # VAE
    num_down_blocks=3,
    latent_channels=4,
    vae_block_out_channels="128,256,512",
    # CELL-FM
    img_mask_ratio=0,
    cond_out_channels="32,64",
    sample_size=64,
    esm_embedding="esmc_600m",
    encoder_hidden_size=1152,
    max_protein_sequence_len=2048,
    encoder_num_hidden_layers=8,
    num_heads=8,
    dim_head=64,
    dropout=0,
    final_dropout=0,
    encoder_patch_size=4,
    # image generator
    img_generator_num_layers=8,
    img_generator_patch_size=2,
    attention_head_dim=64,
    num_attention_heads=18,
    # image decoder
    img_decoder_num_hidden_layers=4,
    img_decoder_hidden_size=512,
    img_decoder_num_heads=8,
    img_decoder_dim_head=64,
    # checkpoints — the OpenCell pair, which must be loaded together
    vae_loadcheck_path=opencell("vae.bin"),
    loadcheck_path=opencell("cellfm_vs.bin"),
    infer=True,
)

# 3.7 GB of CELL-FM weights, plus 2.3 GB for the ESM-C 600M encoder the generator is built
# around — the esm package fetches that when the model is constructed, before the
# checkpoint load, even though the checkpoint carries its own copy of those tensors.
t0 = time.time()
model = CELLFMModel(config=config, loss_fn=UniDiffCriterions)
model.to(DEVICE).eval()
tokenizer = EsmSequenceTokenizer()

# The transformers pin exists to keep this true, so test it rather than trusting a version
# number. When the special-token plumbing moves under esm, mask_token comes back None and
# the failure surfaces much later, inside generation.
if not isinstance(tokenizer.mask_token, str):
    raise RuntimeError(
        f"tokenizer.mask_token is {tokenizer.mask_token!r}, not a string — this build of "
        f"transformers ({transformers.__version__}) does not serve special tokens the way "
        "esm expects, and generation would fail later. Rerun the install cell."
    )

# load_pretrained_weights keeps only the checkpoint keys that match by name AND shape, then
# calls load_state_dict(strict=False) — so a config typo does not raise, it leaves part of
# the model randomly initialised and still produces plausible-looking images. The mismatch
# is reported through loguru at INFO, which scrolls past unread in Colab. Check the tensors
# instead: a module that never received weights still has its init statistics.
_suspicious = [
    name for name, module in
    [("cond_conv", model.net.cond_conv), ("vae", model.vae)]
    if all(p.std().item() == 0 or torch.isnan(p).any() for p in module.parameters()
           if p.dim() > 1)
]
if _suspicious:
    raise RuntimeError(
        f"{', '.join(_suspicious)} look uninitialised — the checkpoint did not load into "
        "them. Almost always a config mismatch against the checkpoint that was downloaded.")

print(f"model {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M params, "
      f"ready on {DEVICE} in {time.time() - t0:.0f}s")

In [ ]:
import io

# OpenCell's own metadata, minus the image_paths column that was 92% of it. One row per
# gene, 1311 of them, carrying the sequence the model was trained on and OpenCell's
# annotation of where the protein actually sits.
GENES = pd.read_csv(opencell("vs_genes.csv"))

# The nucleus every free-form generation is conditioned on: gene ATG7, crop 1 — the same
# one every published OpenCell run uses (cell_fm/tasks/cell_fm/virtual_staining_opencell.py
# :41-45). Holding it fixed is what makes two proteins comparable: any difference between
# their images has to come from the sequence, because nothing else differed.
anchor = torch.from_numpy(np.load(opencell("vs_anchor_nucleus.npy"))).unsqueeze(0).to(DEVICE)

# Seventeen proteins with a real image each; section 2 shows four of them by default.
_ref = np.load(opencell("vs_reference_cells.npz"))
REFERENCE = pd.read_csv(io.StringIO(str(_ref["index"])))


def reference_cell(gene):
    """The baked (nucleus, protein) pair for one reference gene, each (1, 1, 256, 256)."""
    return tuple(
        torch.from_numpy(_ref[f"{gene}/{ch}"].astype(np.float32)).unsqueeze(0).to(DEVICE)
        for ch in ("nucleus", "protein")
    )


fig, ax = plt.subplots(figsize=(2.6, 2.8))
ax.imshow((anchor[0, 0].cpu().numpy() + 1) / 2, cmap="gray", vmin=0, vmax=1)
ax.set_title("the shared conditioning nucleus", fontsize=9, color=NUCLEAR)
ax.set_axis_off()
plt.show()

usable = int((GENES.sequence.str.len() <= config.max_protein_sequence_len).sum())
print(f"{len(GENES):,} OpenCell genes, {usable:,} within the "
      f"{config.max_protein_sequence_len} aa limit, "
      f"{len(REFERENCE)} with a real image baked in")

# These are maximum-intensity projections of confocal stacks, not single optical slices —
# which is why the ER and mitochondrial textures look denser than they would in a slice.
print("images are 256x256 2-D projections; intensities are per-image min-max normalised, "
      "so pattern is comparable between images but brightness is not")

In [ ]:
import difflib
import json
import re
import random
import urllib.parse
import urllib.request

from tqdm.auto import tqdm

AA = set("ACDEFGHIKLMNPQRSTVWY")
# Rough ceiling on tokens per batch item before a 15 GB T4 runs out. The unified encoder
# attends over 1024 image tokens concatenated with the sequence, so cost grows with the
# protein: nls_screening's 26 aa fragments are ~1050 tokens, OpenCell's median 472 aa is
# ~1500, and the 2048 aa cap is ~3070. Inheriting a fixed batch size from that notebook
# would OOM here on a long protein, and an OOM kills the Colab session rather than the cell.
TOKEN_BUDGET = 16000


def seed_everything(seed):
    """Matches cell_fm/pipeline/accelerator/trainer.py, which the offline task calls."""
    torch.cuda.manual_seed_all(seed)
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)


def _safe_batch(requested, seq_len):
    """Cap the batch by the token budget; see TOKEN_BUDGET."""
    per_item = 1024 + seq_len + 2
    return max(1, min(requested, TOKEN_BUDGET // per_item))


def check_length(sequence, what="that sequence"):
    """Refuse a sequence the encoder cannot position.

    The positional table is built with max_protein_sequence_len + 2 = 2050 rows
    (cell_fm_model.py:332-338) and tokenize_sequence adds <cls> and <eos>, so 2048 residues
    is exactly the longest that fits. At 2049 the add at cell_fm_model.py:610 fails with
    "The size of tensor a (2051) must match the size of tensor b (2050)" — thrown from
    inside the model, where it explains nothing. Catch it here instead.

    This is the single choke point: every route to the model goes through generate(), so
    checking here covers the reference panel and any direct call, not just the lookup.
    """
    limit = config.max_protein_sequence_len
    if len(sequence) > limit:
        raise ValueError(
            f"{what} is {len(sequence):,} aa; the model takes at most {limit}. "
            f"35 of OpenCell's 1,311 genes are over that and were left out of training "
            "for the same reason — try a shorter protein, or a domain of this one.")
    return sequence


@torch.no_grad()
def generate(sequence, n_images, cell_img=None, seed=6, batch_size=8, progress=True):
    """Virtually stain n_images cells for one sequence, as (n, 256, 256) in [0, 1].

    cell_img defaults to the shared anchor nucleus. Pass a different one to condition on a
    different cell — which is what the reference panel does, so that a generated image and
    the real image beside it come from the same nucleus.

    Note that the batch size is part of the result, not just its speed: sequence_to_image
    draws its noise as randn(cell_img.shape[0], ...), so re-running with a different batch
    size gives different images at the same seed.
    """
    if cell_img is None:
        cell_img = anchor
    check_length(sequence)
    tokens = encoding.tokenize_sequence(sequence, tokenizer, True).unsqueeze(0).to(DEVICE)
    batch_size = _safe_batch(batch_size, len(sequence))
    seed_everything(seed)

    out, done = [], 0
    bar = tqdm(total=n_images, desc="cells", unit="img", disable=not progress)
    while done < n_images:
        b = min(batch_size, n_images - done)
        try:
            sample = model.sequence_to_image(
                tokens.repeat(b, 1), cell_img.repeat(b, 1, 1, 1), num_steps=NUM_STEPS)
        except torch.cuda.OutOfMemoryError:
            # Halve and retry rather than let the session die; at b == 1 there is nothing
            # left to give back and the error is the honest answer.
            torch.cuda.empty_cache()
            if b == 1:
                raise
            batch_size = max(1, b // 2)
            print(f"  out of memory at batch {b}, retrying at {batch_size}")
            continue
        # the VAE decoder has no final activation, so the output is only nominally [-1, 1]
        out.append(((sample[:, 0] + 1) / 2).clamp(0, 1).cpu().numpy())
        done += b
        bar.update(b)
    bar.close()
    return np.concatenate(out)


def _uniprot(query, exact=True):
    """Look a protein up in UniProt, restricted to reviewed human entries."""
    q = (f"gene_exact:{query}" if exact else query) + " AND organism_id:9606 AND reviewed:true"
    url = ("https://rest.uniprot.org/uniprotkb/search?query=" + urllib.parse.quote(q)
           + "&fields=accession,gene_primary,protein_name,sequence&format=json&size=5")
    with urllib.request.urlopen(url, timeout=30) as response:
        return json.load(response).get("results", [])


def _from_uniprot_hit(hit):
    name = hit.get("proteinDescription", {}).get("recommendedName", {})
    return {
        "gene_name": (hit.get("genes") or [{}])[0].get("geneName", {}).get("value", ""),
        "protein_name": name.get("fullName", {}).get("value"),
        "uniprot": hit["primaryAccession"],
        "locations": None,
        "sequence": hit["sequence"]["value"],
        "in_opencell": False,
    }


def find_protein(query):
    """Resolve a gene name, UniProt accession, protein name or raw sequence to a record.

    OpenCell's table first — those are the proteins the model was trained on, and the only
    ones with a localization annotation and a real image. UniProt after, for anything else.
    """
    query = query.strip()
    if not query:
        raise LookupError("type a gene name, a UniProt accession, or a sequence")

    # a pasted sequence needs no lookup at all
    bare = "".join(query.split()).upper()
    if len(bare) > 30 and set(bare) <= AA:
        return _check_length({"gene_name": "(pasted sequence)", "protein_name": None,
                              "uniprot": "", "locations": None, "sequence": bare,
                              "in_opencell": False})

    hit = GENES[GENES.gene_name.str.casefold() == query.casefold()]
    if hit.empty:
        hit = GENES[GENES.uniprot.str.casefold() == query.casefold()]
    if hit.empty:
        # Word boundary first: a plain substring search for "lamin" also matches
        # "trimethylaminobutyraldehyde dehydrogenase" and, sorted alphabetically, would
        # hand back ALDH9A1 rather than LMNB1. na=False matters too — eleven genes have no
        # protein_name, and indexing on a mask containing NaN raises rather than skipping.
        escaped = re.escape(query)
        hit = GENES[GENES.protein_name.str.contains(
            rf"\b{escaped}", case=False, na=False, regex=True)]
        if hit.empty:
            hit = GENES[GENES.protein_name.str.contains(
                escaped, case=False, na=False, regex=True)]
    if not hit.empty:
        record = hit.iloc[0].to_dict()
        record["in_opencell"] = True
        if len(hit) > 1:
            others = ", ".join(hit.gene_name[1:6])
            print(f"{len(hit)} proteins matched {query!r}; using {record['gene_name']} "
                  f"(also: {others}). Name one of those to pick it instead.")
        return _check_length(record)

    try:
        hits = _uniprot(query)
        if not hits:
            # gene_exact does not match aliases: "GM130" finds nothing, though free text
            # puts GOLGA2 first. Say which one was taken rather than choosing silently.
            hits = _uniprot(query, exact=False)
            if hits:
                print(f"no exact gene match for {query!r}; using "
                      f"{hits[0]['primaryAccession']} — paste an accession to override")
    except Exception as exc:
        raise LookupError(
            f"UniProt lookup for {query!r} failed ({exc}). You can paste the amino-acid "
            "sequence directly instead.") from None

    if not hits:
        close = difflib.get_close_matches(query.upper(), GENES.gene_name.tolist(), 3, 0.6)
        hint = f" Did you mean {', '.join(close)}?" if close else ""
        raise LookupError(f"nothing matched {query!r}.{hint}")

    record = _from_uniprot_hit(hits[0])
    # A UniProt hit may still be an OpenCell gene reached by an alias — round-trip on the
    # accession, which is unique in the table, so those get their annotation and real image.
    same = GENES[GENES.uniprot == record["uniprot"]]
    if not same.empty:
        record = same.iloc[0].to_dict()
        record["in_opencell"] = True
    return _check_length(record)


def _check_length(record):
    """Same limit as check_length, applied at lookup so the error names the protein.

    The offline task skips over-long genes outright (virtual_staining_opencell.py:59-60),
    which is why 1,276 of the 1,311 genes have published generations rather than all of them.
    """
    check_length(record["sequence"], record["gene_name"] or "that sequence")
    return record


def describe(record):
    """A short block about the protein, printed before its images."""
    print(f"{record['gene_name']}  {record['uniprot']}")
    if record.get("protein_name") and not pd.isna(record["protein_name"]):
        print(f"  {record['protein_name']}")
    print(f"  {len(record['sequence'])} aa")
    if record.get("in_opencell"):
        where = record.get("locations")
        print(f"  OpenCell annotation: {where if isinstance(where, str) else 'none'}"
              "  (a database label, not a model output)")
        print("  in the fine-tuning set — the model saw this protein's images")
    else:
        print("  NOT in OpenCell: the model never saw this protein's images, so this is "
              "the out-of-distribution case")

## 2 · Virtual staining

Four OpenCell proteins, painted from their sequences onto the **same nucleus**. 
Nothing changes down the figure except the amino acid sequence, so every difference 
between the rows is the model reading the protein.

| column | what it is |
| --- | --- |
| **input nucleus** | the shared conditioning image, the same one in all four rows |
| **generated** | what the model paints onto it from the sequence alone |
| **real, another cell** | OpenCell's photograph of that protein, for reference |

The real column is a *different cell*, imaged separately — it is there to say what the
protein looks like, not to be compared pixel for pixel against the generation beside it.
Brightness is not comparable either: each image is min-max normalised on its own. Pattern
is what carries.

The four are two matched pairs, which is where the model has something to prove:

- **POLR1A** and **SNRPF** are both nuclear and should not look alike. POLR1A is RNA
  polymerase I, concentrated in the fibrillar centre of the nucleolus; SNRPF is an Sm core
  protein, spread across chromatin.
- **LSM14A** and **DDX6** carry the *same* OpenCell annotation, `big_aggregates`, and are
  both P-body proteins. A model that had only memorised the coarse label would render them
  identically. Look at whether it does.


In [ ]:
# @title Virtual staining { display-mode: "form" }
PANEL_GENES = "POLR1A, SNRPF, LSM14A, DDX6"  # @param {type:"string"}
PANEL_SEED = 6  # @param {type:"integer"}

# Not in the form. The ODE step count is the sampler setting every timing here assumes, and
# lowering it trades image quality for speed close to linearly.
NUM_STEPS = 100

if str(DEVICE) == "cpu":
    raise RuntimeError(
        "No GPU. The sampler runs the ESM-C 600M encoder once per ODE step, so one image "
        "on Colab's CPU takes many minutes rather than seconds. "
        "Runtime > Change runtime type > T4 GPU, then rerun from the top.")

wanted = [g.strip().upper() for g in PANEL_GENES.replace(",", " ").split()]
unknown = [g for g in wanted if g not in set(REFERENCE.gene_name)]
if unknown:
    raise ValueError(f"no baked reference cell for {unknown}. "
                     f"Available: {', '.join(REFERENCE.gene_name)}")

rows = REFERENCE[REFERENCE.gene_name.isin(wanted)].set_index("gene_name").loc[wanted]

# The reference genes are baked, and all seventeen are well inside the length limit (the
# longest, TAF1, is 1,893 aa). Check anyway rather than rely on that: this path reads
# GENES.sequence directly instead of going through find_protein, so nothing else here
# would catch a rebake that let a longer gene in.
for _gene in rows.index:
    check_length(GENES[GENES.gene_name == _gene].iloc[0].sequence, _gene)
fig, axes = plt.subplots(len(rows), 3, figsize=(4.9, 1.62 * len(rows)))
axes = np.atleast_2d(axes)

shared_nucleus = (anchor[0, 0].cpu().numpy() + 1) / 2

t0 = time.time()
for r, (gene, row) in enumerate(rows.iterrows()):
    _, real = reference_cell(gene)
    sequence = GENES[GENES.gene_name == gene].iloc[0].sequence
    # every row conditions on the same anchor, so the sequence is the only thing that
    # differs between them
    generated = generate(sequence, 1, cell_img=anchor, seed=PANEL_SEED, progress=False)[0]

    panels = [("input nucleus", shared_nucleus, "gray"),
              ("generated", generated, "magma"),
              ("real, another cell", (real[0, 0].cpu().numpy() + 1) / 2, "magma")]
    for ax, (label, img, cmap) in zip(axes[r], panels):
        ax.imshow(img, cmap=cmap, vmin=0, vmax=1)
        ax.set_axis_off()
        if r == 0:
            ax.set_title(label, fontsize=8)
    axes[r][0].text(-0.07, 0.5, f"{gene}\n{row.locations}", transform=axes[r][0].transAxes,
                    ha="right", va="center", fontsize=8, color=INK)

fig.tight_layout()
plt.show()
panel = fig

# Every later estimate uses this rather than a hard-coded constant: cost here depends on
# the protein, since the encoder attends over the sequence as well as the image.
SECONDS_PER_IMAGE = (time.time() - t0) / len(rows)
print(f"{len(rows)} images in {time.time() - t0:.0f}s "
      f"({SECONDS_PER_IMAGE:.2f}s per image on this runtime)")

## 3 · Your protein

Type a **gene symbol** (`TOMM20`), a **UniProt accession** (`Q15388`), part of a **protein
name** (`lamin`), or paste a **raw sequence** from the OpenCell library.

In [ ]:
# @title Protein and sampling { display-mode: "form" }
PROTEIN = "TOMM20"  # @param {type:"string"}
N_IMAGES = 8  # @param {type:"integer"}
SEED = 6  # @param {type:"integer"}
BATCH_SIZE = 8  # @param {type:"integer"}

protein = find_protein(PROTEIN)
describe(protein)

batch = _safe_batch(BATCH_SIZE, len(protein["sequence"]))
if batch < BATCH_SIZE:
    print(f"\n  batch capped to {batch} for a {len(protein['sequence'])} aa sequence, "
          "to stay inside a T4's memory")
print(f"\n{N_IMAGES} images, {NUM_STEPS} ODE steps each")
# The panel generates one image per call, so its rate carries no batching benefit and this
# is an upper bound: at batch 8 the run below came in around half of it on an A40.
print(f"estimate: under {N_IMAGES * SECONDS_PER_IMAGE:.0f}s — the rate measured from the "
      "panel above, which ran unbatched, so this errs high")

## 4 · Paint it

The sequence goes in through the ESM-C encoder, the nucleus through the VAE, and the ODE
integrates from noise to a protein channel.

In [ ]:
t0 = time.time()
images = generate(protein["sequence"], N_IMAGES, seed=SEED, batch_size=BATCH_SIZE)
print(f"{len(images)} cells in {time.time() - t0:.0f}s")

n = min(len(images), 8)
fig, axes = plt.subplots(1, n + 1, figsize=(1.45 * (n + 1), 2.1))
axes[0].imshow((anchor[0, 0].cpu().numpy() + 1) / 2, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("nucleus", fontsize=8, color=NUCLEAR)
for i in range(n):
    axes[i + 1].imshow(images[i], cmap="magma", vmin=0, vmax=1)
    axes[i + 1].set_title(f"{i + 1}", fontsize=8)
for ax in axes:
    ax.set_axis_off()

where = protein.get("locations")
subtitle = f" — OpenCell: {where}" if isinstance(where, str) else " — not in OpenCell"
fig.suptitle(f"{protein['gene_name']}{subtitle}", fontsize=10, y=1.02)
plt.show()

# How much the model commits: per-pixel spread across draws relative to the mean signal.
# Low means it puts the protein in the same place every time.
spread = images.std(axis=0).mean() / max(images.mean(), 1e-6)
print(f"mean intensity {images.mean():.3f}, relative spread across draws {spread:.2f}")
if isinstance(where, str):
    print(f"OpenCell annotates {protein['gene_name']} as {where} — a database label shown "
          "for comparison, not something the model was given or produced")

## 5 · Export

| File | What it holds |
| --- | --- |
| `opencell_vs.tif` | the cells from section 4 as a `ZCYX` stack — channel 0 the conditioning nucleus, channel 1 the generated protein. The same layout the offline task writes, so it opens beside the cluster's own outputs |
| `opencell_vs.csv` | one row per image: gene, accession, OpenCell annotation, sequence length, seed |
| `opencell_check.png` | the section 2 panel |

In [ ]:
import tifffile


def export(images, record, panel, stem="opencell_vs"):
    """Write the stack, the table and the panel, then offer all three for download."""
    tif_path, csv_path, png_path = f"{stem}.tif", f"{stem}.csv", "opencell_check.png"

    # Nucleus and generated protein interleaved, matching what the offline task writes to
    # 2d_proj_256_crop_dataset_virtual_staining_same_nucl so the two are interchangeable.
    # clip then round, exactly as save_tif does (virtual_staining_opencell.py:23-29):
    # rounding rather than truncating is what makes the nucleus channel here bit-identical
    # to the cluster's, and the clip matters because astype on a value a hair below zero
    # wraps to 65535 rather than saturating.
    nucleus = (anchor[0, 0].cpu().numpy() + 1) / 2
    stack = np.stack([np.stack([nucleus, img]) for img in images])
    tifffile.imwrite(tif_path, np.round(stack.clip(0, 1) * 65535).astype(np.uint16),
                     imagej=True, metadata={"axes": "ZCYX"})

    where = record.get("locations")
    pd.DataFrame([{
        "image": i + 1,
        "gene_name": record["gene_name"],
        "uniprot": record["uniprot"],
        "protein_name": record.get("protein_name"),
        "opencell_locations": where if isinstance(where, str) else "",
        "in_opencell": record.get("in_opencell", False),
        "sequence_length": len(record["sequence"]),
        "num_steps": NUM_STEPS,
        "seed": SEED,
    } for i in range(len(images))]).to_csv(csv_path, index=False)

    panel.savefig(png_path, dpi=200, bbox_inches="tight", facecolor=SURFACE)
    download_buttons(tif_path, csv_path, png_path)
    return tif_path, csv_path, png_path


def download_buttons(*paths):
    """One button per file, rather than firing the downloads the moment the cell runs.

    The Output widget is load-bearing: files.download runs JavaScript in a live output
    context, and a click handler has none of its own, so calling it straight from on_click
    silently does nothing.
    """
    for path in paths:
        print(f"{path}  {os.path.getsize(path) / 1e6:.2f} MB")

    try:
        import ipywidgets as widgets
        from google.colab import files
        from IPython.display import display
    except ImportError:
        print("\nno download button outside Colab — the files are in the working directory")
        return

    sink = widgets.Output()

    def fetch(path):
        with sink:
            files.download(path)

    buttons = [
        widgets.Button(description=f"Download {os.path.basename(p)}",
                       icon="download", layout=widgets.Layout(width="auto"))
        for p in paths
    ]
    for button, path in zip(buttons, paths):
        button.on_click(lambda _, p=path: fetch(p))

    display(widgets.HBox(buttons), sink)


_ = export(images, protein, panel)